## 7 — Databricks Cleanup

Drops all tables and views in the configured Unity Catalog schema.

**Preserved (NOT touched):**
- External volume + contents (raw `.dat` datasets, logs, results, JMeter test files)
- Storage credential, external location
- Catalog, schema themselves

**Dropped:**
- Every table / view under `{dbx_catalog}.{dbx_schema}`


In [ ]:
# --- Create Widgets ---
dbutils.widgets.removeAll()
dbutils.widgets.text("dbx_catalog", "tpcds_sf_catalog", "Databricks Catalog")
dbutils.widgets.text("dbx_schema", "tpcds", "Databricks Schema")
dbutils.widgets.text("table_prefix", "", "Table Prefix filter (optional; blank = drop ALL)")
dbutils.widgets.dropdown("dry_run", "false", ["true", "false"], "Dry Run")

In [ ]:
params = dbutils.widgets.getAll()

catalog = params["dbx_catalog"]
schema = params["dbx_schema"]
prefix = params.get("table_prefix", "").strip()
dry_run = params.get("dry_run", "false").lower() == "true"

if not catalog or not schema:
    raise RuntimeError("Validation Error: dbx_catalog and dbx_schema are required.")

print(f"Target: {catalog}.{schema}")
print(f"Prefix filter: {prefix!r}" if prefix else "Prefix filter: (none — every table will be dropped)")
print(f"Dry run: {dry_run}")

In [ ]:
# Discover tables and views in the schema
spark.sql(f"USE CATALOG {catalog}")
tables_df = spark.sql(f"SHOW TABLES IN {catalog}.{schema}").collect()
views_df = spark.sql(f"SHOW VIEWS IN {catalog}.{schema}").collect()

view_names = {row["viewName"] for row in views_df}

targets = []  # list of (name, kind)
for row in tables_df:
    name = row["tableName"]
    # SHOW TABLES includes views on some DBR versions; keep kind consistent
    kind = "VIEW" if name in view_names else "TABLE"
    if prefix and not name.startswith(prefix):
        continue
    # Skip temporary / isTemporary rows
    if row.asDict().get("isTemporary", False):
        continue
    if name.lower().startswith("results_"):
        continue  # preserve benchmark-results tables
    targets.append((name, kind))

# Any view not already picked up above (in case SHOW TABLES omits views on this runtime)
for v in view_names:
    if not any(n == v for n, _ in targets):
        if prefix and not v.startswith(prefix):
            continue
        if v.lower().startswith("results_"):
            continue  # preserve benchmark-results views
        targets.append((v, "VIEW"))

# Drop views first, tables second — views may depend on tables
targets.sort(key=lambda t: (t[1] != "VIEW", t[0]))

print(f"Found {len(targets)} object(s) to drop")
for name, kind in targets:
    print(f"  [{kind}] {catalog}.{schema}.{name}")

In [ ]:
# Execute drops
dropped = 0
failed = []
for name, kind in targets:
    stmt = f"DROP {kind} IF EXISTS `{catalog}`.`{schema}`.`{name}`"
    if dry_run:
        print(f"DRY RUN -> {stmt}")
        continue
    try:
        spark.sql(stmt)
        dropped += 1
        print(f"dropped [{kind}] {name}")
    except Exception as e:
        failed.append((name, kind, str(e)))
        print(f"FAILED [{kind}] {name}: {e}")

print(f"\nSummary: {dropped}/{len(targets)} dropped, {len(failed)} failed")
if failed:
    for n, k, err in failed:
        print(f"  - {k} {n}: {err[:200]}")
    raise RuntimeError(f"Cleanup completed with {len(failed)} failure(s)")

In [ ]:
# Sanity check: the volume, catalog, and schema are still intact
print("Catalog still exists:")
spark.sql(f"DESCRIBE CATALOG {catalog}").show(truncate=False)
print("\nSchema still exists:")
spark.sql(f"DESCRIBE SCHEMA {catalog}.{schema}").show(truncate=False)
print("\nVolume contents (raw data + test files should be intact):")
try:
    items = dbutils.fs.ls(f"/Volumes/{catalog}/{schema}/tpcds_raw_data/")
    for item in items:
        print(f"  {item.name}")
except Exception as e:
    print(f"  (no standard volume found: {e})")